In [1]:
import pandas as pd
import metapredict

# 1. Get IDR intervals

In [3]:
# Reading in all TF sequences
TF_seqs = pd.read_csv("../output/lambert_TFs_10-21-24_with_DBD_coords.csv", index_col = 0)
TF_seqs

,id,ProteinSeq,DBD_coords_merged
0,sp|A0A087WUV0|ZN892_HUMAN Zinc finger protein ...,MEPEGRGSLFEDSDLLHAGNPKENDVTAVLLTPGSQELMIRDMAEA...,"[[221, 243], [249, 271], [277, 299], [305, 327..."
1,sp|A0AVK6|E2F8_HUMAN Transcription factor E2F8...,MENEKENLFCEPHKRGLMKTPLKESTTANIVLAEIQPDFGPLTTPT...,"[[114, 182], [262, 347]]"
2,sp|A0PJY2|FEZF1_HUMAN Fez family zinc finger p...,MDSSCHNATTKMLATAPARGNMMSTSKPLAFSIERIMARTPEPKAL...,"[[260, 282], [288, 310], [316, 338], [344, 366..."
3,sp|A1A519|F170A_HUMAN Protein FAM170A OS=Homo ...,MKRRQKRKHLENEESQETAEKGGGMSKSQEDALQPGSTRVAKGWSQ...,"[[1, 330]]"
4,sp|A1YPR0|ZBT7C_HUMAN Zinc finger and BTB doma...,MANDIDELIGIPFPNHSSEVLCSLNEQRHDGLLCDVLLVVQEQEYR...,"[[364, 386], [392, 414], [420, 442], [448, 469]]"
...,...,...,...
1608,sp|Q9Y6Q9|NCOA3_HUMAN Nuclear receptor coactiv...,MSGLGENLDPLASDSRKRKLPCDTPGQGLTCSGEKRRREQESKYIE...,"[[31, 83]]"
1609,sp|Q9Y6R6|Z780B_HUMAN Zinc finger protein 780B...,MVHGSVTFRDVAIDFSQEEWECLQPDQRTLYRDVMLENYSHLISLG...,"[[165, 187], [193, 215], [221, 243], [249, 271..."
1610,sp|Q9Y6X0|SETBP_HUMAN SET-binding protein OS=H...,MESRETLSSSRQRGGESDFLPVSSAKPPAAPGCAGEPLLSTPGPGK...,"[[583, 596], [1015, 1027], [1450, 1462]]"
1611,sp|Q9Y6X8|ZHX2_HUMAN Zinc fingers and homeobox...,MASKRKSTTPCMVRTSQVVEQDVPEEVDRAKEKGIGTPQPDVAKDS...,"[[78, 101], [110, 133], [271, 317], [442, 496]..."


In [92]:
# Running metapredict on each sequence to get IDR intervals
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

def process_sequence(seq):
    output = [[start + 1, end + 1] for start, end in metapredict.predict_disorder_domains(seq).disordered_domain_boundaries]
    return "NA-NA" if len(output) == 0 else str(output)

results = []
with ThreadPoolExecutor() as executor:
    for res in tqdm(executor.map(process_sequence, TF_seqs["ProteinSeq"]), total=len(TF_seqs)):
        results.append(res)

TF_seqs["IDR_coords_merged"] = results


100%|██████████| 1613/1613 [00:07<00:00, 212.77it/s]


In [93]:
TF_seqs["uniprotID"] = TF_seqs["id"].str.split("|").str[1]
TF_seqs

,id,ProteinSeq,DBD_coords_merged,IDR_coords_merged,uniprotID,IDR_coords
0,sp|A0A087WUV0|ZN892_HUMAN Zinc finger protein ...,MEPEGRGSLFEDSDLLHAGNPKENDVTAVLLTPGSQELMIRDMAEA...,"[[221, 243], [249, 271], [277, 299], [305, 327...","[[1, 216], [498, 523]]",A0A087WUV0,"1-216,498-523"
1,sp|A0AVK6|E2F8_HUMAN Transcription factor E2F8...,MENEKENLFCEPHKRGLMKTPLKESTTANIVLAEIQPDFGPLTTPT...,"[[114, 182], [262, 347]]","[[1, 115], [337, 868]]",A0AVK6,"1-115,337-868"
2,sp|A0PJY2|FEZF1_HUMAN Fez family zinc finger p...,MDSSCHNATTKMLATAPARGNMMSTSKPLAFSIERIMARTPEPKAL...,"[[260, 282], [288, 310], [316, 338], [344, 366...","[[1, 220], [422, 476]]",A0PJY2,"1-220,422-476"
3,sp|A1A519|F170A_HUMAN Protein FAM170A OS=Homo ...,MKRRQKRKHLENEESQETAEKGGGMSKSQEDALQPGSTRVAKGWSQ...,"[[1, 330]]","[[1, 219], [269, 331]]",A1A519,"1-219,269-331"
4,sp|A1YPR0|ZBT7C_HUMAN Zinc finger and BTB doma...,MANDIDELIGIPFPNHSSEVLCSLNEQRHDGLLCDVLLVVQEQEYR...,"[[364, 386], [392, 414], [420, 442], [448, 469]]","[[128, 364], [471, 620]]",A1YPR0,"128-364,471-620"
...,...,...,...,...,...,...
1608,sp|Q9Y6Q9|NCOA3_HUMAN Nuclear receptor coactiv...,MSGLGENLDPLASDSRKRKLPCDTPGQGLTCSGEKRRREQESKYIE...,"[[31, 83]]","[[1, 40], [86, 108], [177, 193], [355, 1425]]",Q9Y6Q9,"1-40,86-108,177-193,355-1425"
1609,sp|Q9Y6R6|Z780B_HUMAN Zinc finger protein 780B...,MVHGSVTFRDVAIDFSQEEWECLQPDQRTLYRDVMLENYSHLISLG...,"[[165, 187], [193, 215], [221, 243], [249, 271...","[[64, 161], [809, 834]]",Q9Y6R6,"64-161,809-834"
1610,sp|Q9Y6X0|SETBP_HUMAN SET-binding protein OS=H...,MESRETLSSSRQRGGESDFLPVSSAKPPAAPGCAGEPLLSTPGPGK...,"[[583, 596], [1015, 1027], [1450, 1462]]","[[1, 1400], [1437, 1597]]",Q9Y6X0,"1-1400,1437-1597"
1611,sp|Q9Y6X8|ZHX2_HUMAN Zinc fingers and homeobox...,MASKRKSTTPCMVRTSQVVEQDVPEEVDRAKEKGIGTPQPDVAKDS...,"[[78, 101], [110, 133], [271, 317], [442, 496]...","[[1, 76], [150, 283], [334, 447], [589, 640], ...",Q9Y6X8,"1-76,150-283,334-447,589-640,695-838"


In [94]:
all_TFs_table = pd.read_csv("../soto_analysis/outputs/all_TFs_table_proteins.txt", sep = "\t", index_col = 0)
all_TFs_table

,1,2,uniprotID,ENSG,ENST,DBD_coords,AD_coords,RD_coords,Bif_coords,length
0,NaN,NaN,A0A087WUV0,NaN,ENST00000425953,"221-243,249-271,277-299,305-327,333-355,361-38...",NaN,NaN,NaN,1
1,NaN,NaN,A0AVK6,NaN,ENST00000250024,"114-182,262-347",NaN,NaN,NaN,1
2,NaN,NaN,A1YPR0,NaN,ENST00000535628,"364-386,392-414,420-442,448-469",NaN,NaN,NaN,1
3,NaN,NaN,A2RRD8,NaN,ENST00000391781,"161-183,189-211,217-239,245-267,273-295,301-32...",NaN,NaN,NaN,1
4,NaN,NaN,A2RU54,NaN,ENST00000339992,150-206,NaN,NaN,NaN,1
...,...,...,...,...,...,...,...,...,...,...
1585,NaN,NaN,Q9Y692,NaN,ENST00000294409,89-165,422-573,NaN,NaN,1
1586,NaN,NaN,Q9Y6Q3,NaN,ENST00000374227,"293-315,321-343,377-399,405-427,433-455,461-48...",NaN,NaN,NaN,1
1587,NaN,NaN,Q9Y6Q9,NaN,ENST00000371998,31-83,621-1424,NaN,NaN,1
1588,NaN,NaN,Q9Y6X0,NaN,ENST00000649279,"583-596,1015-1027,1450-1462",NaN,NaN,NaN,1


In [100]:
# Converting the output from metapredict to output necessary for Soto script
import ast

def convert_ranges_from_string(input_string):
    # Convert the string to a list of lists
    input_list = ast.literal_eval(input_string)
    
    # Convert the list of lists into the desired string format
    return ','.join([f"{start}-{end}" for start, end in input_list])

IDR_coords = []
for i in TF_seqs["IDR_coords_merged"]:
    print(i)
    if i == "NA-NA":
        IDR_coords.append("NA-NA")
    else:
        IDR_coords.append(convert_ranges_from_string(i))

TF_seqs["IDR_coords"] = IDR_coords
TF_seqs

[[1, 216], [498, 523]]
[[1, 115], [337, 868]]
[[1, 220], [422, 476]]
[[1, 219], [269, 331]]
[[128, 364], [471, 620]]
[[51, 156]]
[[1, 158], [224, 274]]
[[1, 56], [126, 1350]]
[[1, 138], [194, 302]]
[[1, 14], [131, 243]]
[[1, 162], [206, 345], [501, 690], [932, 1005]]
[[1, 20], [79, 110], [173, 192]]
[[1, 48], [123, 352], [472, 497]]
[[1, 249], [579, 633]]
[[1, 235], [308, 358]]
[[1, 125]]
[[1, 144], [195, 266]]
[[1, 83], [137, 208], [266, 316]]
[[1, 57], [142, 351]]
[[1, 109], [161, 532]]
[[63, 450], [658, 671]]
[[70, 180], [544, 573]]
[[1, 95], [221, 413]]
[[74, 100], [162, 205]]
[[1, 18], [72, 149], [367, 417]]
[[1, 187], [241, 344]]
[[70, 242], [1208, 1253]]
[[1, 30], [90, 264]]
[[65, 173]]
[[1, 14], [67, 212], [482, 496]]
[[1, 32], [85, 168], [373, 406]]
[[55, 195]]
[[1, 21], [84, 319]]
NA-NA
[[1, 133], [234, 421]]
[[1, 109], [221, 252]]
[[60, 170]]
[[63, 166], [470, 500]]
[[67, 167]]
[[58, 231]]
[[1, 86], [159, 493]]
[[68, 143]]
[[1, 28], [93, 278], [373, 449]]
[[1, 23], [120, 292

,id,ProteinSeq,DBD_coords_merged,IDR_coords_merged,uniprotID,IDR_coords
0,sp|A0A087WUV0|ZN892_HUMAN Zinc finger protein ...,MEPEGRGSLFEDSDLLHAGNPKENDVTAVLLTPGSQELMIRDMAEA...,"[[221, 243], [249, 271], [277, 299], [305, 327...","[[1, 216], [498, 523]]",A0A087WUV0,"1-216,498-523"
1,sp|A0AVK6|E2F8_HUMAN Transcription factor E2F8...,MENEKENLFCEPHKRGLMKTPLKESTTANIVLAEIQPDFGPLTTPT...,"[[114, 182], [262, 347]]","[[1, 115], [337, 868]]",A0AVK6,"1-115,337-868"
2,sp|A0PJY2|FEZF1_HUMAN Fez family zinc finger p...,MDSSCHNATTKMLATAPARGNMMSTSKPLAFSIERIMARTPEPKAL...,"[[260, 282], [288, 310], [316, 338], [344, 366...","[[1, 220], [422, 476]]",A0PJY2,"1-220,422-476"
3,sp|A1A519|F170A_HUMAN Protein FAM170A OS=Homo ...,MKRRQKRKHLENEESQETAEKGGGMSKSQEDALQPGSTRVAKGWSQ...,"[[1, 330]]","[[1, 219], [269, 331]]",A1A519,"1-219,269-331"
4,sp|A1YPR0|ZBT7C_HUMAN Zinc finger and BTB doma...,MANDIDELIGIPFPNHSSEVLCSLNEQRHDGLLCDVLLVVQEQEYR...,"[[364, 386], [392, 414], [420, 442], [448, 469]]","[[128, 364], [471, 620]]",A1YPR0,"128-364,471-620"
...,...,...,...,...,...,...
1608,sp|Q9Y6Q9|NCOA3_HUMAN Nuclear receptor coactiv...,MSGLGENLDPLASDSRKRKLPCDTPGQGLTCSGEKRRREQESKYIE...,"[[31, 83]]","[[1, 40], [86, 108], [177, 193], [355, 1425]]",Q9Y6Q9,"1-40,86-108,177-193,355-1425"
1609,sp|Q9Y6R6|Z780B_HUMAN Zinc finger protein 780B...,MVHGSVTFRDVAIDFSQEEWECLQPDQRTLYRDVMLENYSHLISLG...,"[[165, 187], [193, 215], [221, 243], [249, 271...","[[64, 161], [809, 834]]",Q9Y6R6,"64-161,809-834"
1610,sp|Q9Y6X0|SETBP_HUMAN SET-binding protein OS=H...,MESRETLSSSRQRGGESDFLPVSSAKPPAAPGCAGEPLLSTPGPGK...,"[[583, 596], [1015, 1027], [1450, 1462]]","[[1, 1400], [1437, 1597]]",Q9Y6X0,"1-1400,1437-1597"
1611,sp|Q9Y6X8|ZHX2_HUMAN Zinc fingers and homeobox...,MASKRKSTTPCMVRTSQVVEQDVPEEVDRAKEKGIGTPQPDVAKDS...,"[[78, 101], [110, 133], [271, 317], [442, 496]...","[[1, 76], [150, 283], [334, 447], [589, 640], ...",Q9Y6X8,"1-76,150-283,334-447,589-640,695-838"


In [231]:
all_TFs_table_with_IDR = pd.merge(all_TFs_table, TF_seqs[["uniprotID", "IDR_coords"]], how='left')
#.to_csv("../soto_analysis/outputs/all_TFs_table_proteins_with_IDR.txt", sep = "\t")

In [232]:
all_TFs_table_with_IDR

,1,2,uniprotID,ENSG,ENST,DBD_coords,AD_coords,RD_coords,Bif_coords,length,IDR_coords
0,NaN,NaN,A0A087WUV0,NaN,ENST00000425953,"221-243,249-271,277-299,305-327,333-355,361-38...",NaN,NaN,NaN,1,"1-216,498-523"
1,NaN,NaN,A0AVK6,NaN,ENST00000250024,"114-182,262-347",NaN,NaN,NaN,1,"1-115,337-868"
2,NaN,NaN,A1YPR0,NaN,ENST00000535628,"364-386,392-414,420-442,448-469",NaN,NaN,NaN,1,"128-364,471-620"
3,NaN,NaN,A2RRD8,NaN,ENST00000391781,"161-183,189-211,217-239,245-267,273-295,301-32...",NaN,NaN,NaN,1,51-156
4,NaN,NaN,A2RU54,NaN,ENST00000339992,150-206,NaN,NaN,NaN,1,"1-158,224-274"
...,...,...,...,...,...,...,...,...,...,...,...
1585,NaN,NaN,Q9Y692,NaN,ENST00000294409,89-165,422-573,NaN,NaN,1,"1-90,183-258,348-574"
1586,NaN,NaN,Q9Y6Q3,NaN,ENST00000374227,"293-315,321-343,377-399,405-427,433-455,461-48...",NaN,NaN,NaN,1,"1-290,618-631"
1587,NaN,NaN,Q9Y6Q9,NaN,ENST00000371998,31-83,621-1424,NaN,NaN,1,"1-40,86-108,177-193,355-1425"
1588,NaN,NaN,Q9Y6X0,NaN,ENST00000649279,"583-596,1015-1027,1450-1462",NaN,NaN,NaN,1,"1-1400,1437-1597"


# 2. Subtract AD intervals from IDR

In [217]:
# Need to get AD as bed 
all_TFs_table_with_IDR_AD = all_TFs_table_with_IDR[["uniprotID", "AD_coords"]]
all_TFs_table_with_IDR_AD["AD_coords"] = all_TFs_table_with_IDR_AD["AD_coords"].str.split(",")
all_TFs_table_with_IDR_AD = all_TFs_table_with_IDR_AD.explode("AD_coords")
all_TFs_table_with_IDR_AD = all_TFs_table_with_IDR_AD.dropna()
all_TFs_table_with_IDR_AD["start"] = all_TFs_table_with_IDR_AD["AD_coords"].str.split("-").str[0].astype(int)
all_TFs_table_with_IDR_AD["end"] = all_TFs_table_with_IDR_AD["AD_coords"].str.split("-").str[1].astype(int)
all_TFs_table_with_IDR_AD = all_TFs_table_with_IDR_AD.drop(columns= "AD_coords")
all_TFs_table_with_IDR_AD = all_TFs_table_with_IDR_AD[all_TFs_table_with_IDR_AD["start"] != "NA"]
all_TFs_table_with_IDR_AD

/var/folders/hw/xx051vr9457c7lrngf2mypgr0000gn/T/ipykernel_81919/3605998393.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  all_TFs_table_with_IDR_AD["AD_coords"] = all_TFs_table_with_IDR_AD["AD_coords"].str.split(",")


,uniprotID,start,end
13,A6NJG6,142,315
27,A8MTJ6,369,420
33,A8MYZ6,382,491
45,O00570,125,391
48,O14627,32,131
...,...,...,...
1582,Q9Y5R6,172,251
1585,Q9Y692,422,573
1587,Q9Y6Q9,621,1424
1589,Q9Y6Y1,702,841


In [218]:
# IDR as bed
all_TFs_table_with_IDR_IDR = all_TFs_table_with_IDR[["uniprotID", "IDR_coords"]]
all_TFs_table_with_IDR_IDR["IDR_coords"] = all_TFs_table_with_IDR_IDR["IDR_coords"].str.split(",")
all_TFs_table_with_IDR_IDR = all_TFs_table_with_IDR_IDR.explode("IDR_coords")
all_TFs_table_with_IDR_IDR = all_TFs_table_with_IDR_IDR.dropna()
all_TFs_table_with_IDR_IDR = all_TFs_table_with_IDR_IDR[all_TFs_table_with_IDR_IDR["IDR_coords"] != "NA-NA"]
all_TFs_table_with_IDR_IDR["start"] = all_TFs_table_with_IDR_IDR["IDR_coords"].str.split("-").str[0].astype(int)
all_TFs_table_with_IDR_IDR["end"] = all_TFs_table_with_IDR_IDR["IDR_coords"].str.split("-").str[1].astype(int)
all_TFs_table_with_IDR_IDR = all_TFs_table_with_IDR_IDR.drop(columns= "IDR_coords")
all_TFs_table_with_IDR_IDR

/var/folders/hw/xx051vr9457c7lrngf2mypgr0000gn/T/ipykernel_81919/175853898.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  all_TFs_table_with_IDR_IDR["IDR_coords"] = all_TFs_table_with_IDR_IDR["IDR_coords"].str.split(",")


,uniprotID,start,end
0,A0A087WUV0,1,216
0,A0A087WUV0,498,523
1,A0AVK6,1,115
1,A0AVK6,337,868
2,A1YPR0,128,364
...,...,...,...
1589,Q9Y6Y1,1,55
1589,Q9Y6Y1,245,863
1589,Q9Y6Y1,993,1035
1589,Q9Y6Y1,1171,1515


In [219]:
all_TFs_table_with_IDR_AD.to_csv("../data/all_TFs_table_with_IDR_AD.bed", sep="\t", index=False, header=None)
all_TFs_table_with_IDR_IDR.to_csv("../data/all_TFs_table_with_IDR_IDR.bed", sep="\t", index=False, header=None)

In [ ]:
# Remove AD intervals from IDR
! bedtools subtract -a ../data/all_TFs_table_with_IDR_IDR.bed -b ../data/all_TFs_table_with_IDR_AD.bed > ../data/IDR_unique.bed

In [221]:
# Removed
IDR_AD_removed = pd.read_csv("../data/IDR_unique.bed", sep = "\t", header = None)
IDR_AD_removed.columns = ["uniprotID", "start", "end"]
IDR_AD_removed

,uniprotID,start,end
0,A0A087WUV0,1,216
1,A0A087WUV0,498,523
2,A0AVK6,1,115
3,A0AVK6,337,868
4,A1YPR0,128,364
...,...,...,...
3730,Q9Y6Y1,501,702
3731,Q9Y6Y1,841,863
3732,Q9Y6Y1,993,1035
3733,Q9Y6Y1,1171,1515


In [226]:
# Formatting output
IDR_AD_removed["non_AD_IDR_coords"] = IDR_AD_removed["start"].astype(str) + "-" + IDR_AD_removed["end"].astype(str)
IDR_AD_removed = IDR_AD_removed.drop(columns=["start", "end"])
IDR_AD_removed_grouped = IDR_AD_removed.groupby("uniprotID").agg(lambda x: ",".join(x))
IDR_AD_removed_grouped = IDR_AD_removed_grouped.reset_index()
IDR_AD_removed_grouped

,uniprotID,non_AD_IDR_coords
0,A0A087WUV0,"1-216,498-523"
1,A0AVK6,"1-115,337-868"
2,A0PJY2,"1-220,422-476"
3,A1A519,"1-219,269-331"
4,A1YPR0,"128-364,471-620"
...,...,...
1572,Q9Y6Q9,"1-40,86-108,177-193,355-621,1424-1425"
1573,Q9Y6R6,"64-161,809-834"
1574,Q9Y6X0,"1-1400,1437-1597"
1575,Q9Y6X8,"1-76,150-283,334-447,589-640,695-838"


In [228]:
# Merging back with the output table
all_TFs_table_with_IDR_minus_AD = pd.merge(all_TFs_table_with_IDR, IDR_AD_removed_grouped, how='left')

all_TFs_table_with_IDR_minus_AD

,1,2,uniprotID,ENSG,ENST,DBD_coords,AD_coords,RD_coords,Bif_coords,length,IDR_coords,non_AD_IDR_coords
0,NaN,NaN,A0A087WUV0,NaN,ENST00000425953,"221-243,249-271,277-299,305-327,333-355,361-38...",NaN,NaN,NaN,1,"1-216,498-523","1-216,498-523"
1,NaN,NaN,A0AVK6,NaN,ENST00000250024,"114-182,262-347",NaN,NaN,NaN,1,"1-115,337-868","1-115,337-868"
2,NaN,NaN,A1YPR0,NaN,ENST00000535628,"364-386,392-414,420-442,448-469",NaN,NaN,NaN,1,"128-364,471-620","128-364,471-620"
3,NaN,NaN,A2RRD8,NaN,ENST00000391781,"161-183,189-211,217-239,245-267,273-295,301-32...",NaN,NaN,NaN,1,51-156,51-156
4,NaN,NaN,A2RU54,NaN,ENST00000339992,150-206,NaN,NaN,NaN,1,"1-158,224-274","1-158,224-274"
...,...,...,...,...,...,...,...,...,...,...,...,...
1585,NaN,NaN,Q9Y692,NaN,ENST00000294409,89-165,422-573,NaN,NaN,1,"1-90,183-258,348-574","1-90,183-258,348-422,573-574"
1586,NaN,NaN,Q9Y6Q3,NaN,ENST00000374227,"293-315,321-343,377-399,405-427,433-455,461-48...",NaN,NaN,NaN,1,"1-290,618-631","1-290,618-631"
1587,NaN,NaN,Q9Y6Q9,NaN,ENST00000371998,31-83,621-1424,NaN,NaN,1,"1-40,86-108,177-193,355-1425","1-40,86-108,177-193,355-621,1424-1425"
1588,NaN,NaN,Q9Y6X0,NaN,ENST00000649279,"583-596,1015-1027,1450-1462",NaN,NaN,NaN,1,"1-1400,1437-1597","1-1400,1437-1597"


In [230]:
# Inspecting rows which changed
all_TFs_table_with_IDR_minus_AD[all_TFs_table_with_IDR_minus_AD["IDR_coords"] != all_TFs_table_with_IDR_minus_AD["non_AD_IDR_coords"]].head(10)

,1,2,uniprotID,ENSG,ENST,DBD_coords,AD_coords,RD_coords,Bif_coords,length,IDR_coords,non_AD_IDR_coords
13,NaN,NaN,A6NJG6,NaN,ENST00000334384,79-135,142-315,NaN,NaN,1,"1-83,137-208,266-316","1-83,137-142,315-316"
27,NaN,NaN,A8MTJ6,NaN,ENST00000428390,145-234,369-420,NaN,NaN,1,"1-133,234-421","1-133,234-369,420-421"
33,NaN,NaN,A8MYZ6,NaN,ENST00000641094,89-178,382-491,NaN,NaN,1,"1-86,159-493","1-86,159-382,491-493"
45,NaN,NaN,O00570,NaN,ENST00000330949,51-119,125-391,NaN,NaN,1,"1-57,128-392","1-57,391-392"
48,NaN,NaN,O14627,NaN,ENST00000373514,174-230,32-131,NaN,NaN,1,"1-160,237-285","1-32,131-160,237-285"
50,NaN,NaN,O14813,NaN,ENST00000298231,91-147,2-81,NaN,NaN,1,"1-86,150-285","1-2,81-86,150-285"
57,NaN,NaN,O15353,NaN,ENST00000226247,271-361,"482-601,152-231",NaN,NaN,1,"1-272,372-649","1-152,231-272,372-482,601-649"
58,NaN,NaN,O15370,NaN,ENST00000342665,40-108,232-315,NaN,NaN,1,"1-39,107-316","1-39,107-232,315-316"
59,NaN,NaN,O15391,NaN,ENST00000429584,"254-278,283-305,311-335,341-365",2-102,NaN,NaN,1,1-253,"1-2,102-253"
64,NaN,NaN,O43186,NaN,ENST00000221996,40-96,"126-174,192-291",NaN,NaN,1,"1-42,97-300","1-42,97-126,174-192,291-300"


In [233]:
# Drop IDR_coords
all_TFs_table_with_IDR_minus_AD = all_TFs_table_with_IDR_minus_AD.drop(columns="IDR_coords")
all_TFs_table_with_IDR_minus_AD

,1,2,uniprotID,ENSG,ENST,DBD_coords,AD_coords,RD_coords,Bif_coords,length,non_AD_IDR_coords
0,NaN,NaN,A0A087WUV0,NaN,ENST00000425953,"221-243,249-271,277-299,305-327,333-355,361-38...",NaN,NaN,NaN,1,"1-216,498-523"
1,NaN,NaN,A0AVK6,NaN,ENST00000250024,"114-182,262-347",NaN,NaN,NaN,1,"1-115,337-868"
2,NaN,NaN,A1YPR0,NaN,ENST00000535628,"364-386,392-414,420-442,448-469",NaN,NaN,NaN,1,"128-364,471-620"
3,NaN,NaN,A2RRD8,NaN,ENST00000391781,"161-183,189-211,217-239,245-267,273-295,301-32...",NaN,NaN,NaN,1,51-156
4,NaN,NaN,A2RU54,NaN,ENST00000339992,150-206,NaN,NaN,NaN,1,"1-158,224-274"
...,...,...,...,...,...,...,...,...,...,...,...
1585,NaN,NaN,Q9Y692,NaN,ENST00000294409,89-165,422-573,NaN,NaN,1,"1-90,183-258,348-422,573-574"
1586,NaN,NaN,Q9Y6Q3,NaN,ENST00000374227,"293-315,321-343,377-399,405-427,433-455,461-48...",NaN,NaN,NaN,1,"1-290,618-631"
1587,NaN,NaN,Q9Y6Q9,NaN,ENST00000371998,31-83,621-1424,NaN,NaN,1,"1-40,86-108,177-193,355-621,1424-1425"
1588,NaN,NaN,Q9Y6X0,NaN,ENST00000649279,"583-596,1015-1027,1450-1462",NaN,NaN,NaN,1,"1-1400,1437-1597"


In [234]:
all_TFs_table_with_IDR_minus_AD.to_csv("../data/all_TFs_table_proteins_with_IDR.txt")